In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, current_timestamp, regexp_extract
from pyspark.sql.types import DoubleType, StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime
import json

class PeopleDataQualityValidator:
    def __init__(self, spark_session, run_id, pipeline_name):
        self.spark = spark_session
        self.run_id = run_id
        self.pipeline_name = pipeline_name
        self.results = []
 
    def log_result(self, table_name, check_name, status, message, invalid_count=0):
        result = {
            "run_id": self.run_id,
            "pipeline_name": self.pipeline_name,
            "test_timestamp": datetime.now(),
            "table_name": table_name,
            "test_check": check_name,
            "record_count": invalid_count,
            "message": message,
            "status": status
        }
        self.results.append(result)
        print(f"[{status}] {table_name} - {check_name}: {message} (Count: {invalid_count})")
 
    def check_column_types(self, df, table_name, expected_types):
        """
        Checks if columns have the expected data types.
        expected_types: dict { 'column_name': 'expected_type_string' } 
        """
        print(f"Running Column Type checks on {table_name}...")
        actual_types = {f.name: f.dataType.typeName() for f in df.schema}
        
        for col_name, expected_type in expected_types.items():
            if col_name not in actual_types:
                # Try case insensitive match
                found = False
                for actual_col in actual_types:
                    if actual_col.lower() == col_name.lower():
                        col_name = actual_col
                        found = True
                        break
                if not found:
                    self.log_result(table_name, f"Type Check - {col_name}", "ERROR", f"Column {col_name} not found in schema")
                    continue
            
            actual_type = actual_types[col_name]
            if actual_type != expected_type:
                 self.log_result(table_name, f"Type Check - {col_name}", "FAIL", f"Expected {expected_type}, found {actual_type}")
            else:
                 self.log_result(table_name, f"Type Check - {col_name}", "PASS", f"Type match: {actual_type}")
 
    def check_nulls(self, df, table_name, columns):
        """Checks for null values in critical columns."""
        print(f"Running Null checks on {table_name}...")
        for column in columns:
            if column not in df.columns:
                self.log_result(table_name, f"Null Check - {column}", "ERROR", f"Column {column} not found")
                continue
            
            null_count = df.filter(col(column).isNull() | (col(column) == "")).count()
            if null_count > 0:
                self.log_result(table_name, f"Null Check - {column}", "FAIL", f"Found {null_count} null/empty values", null_count)
            else:
                self.log_result(table_name, f"Null Check - {column}", "PASS", "No null values found")
 
    def check_uniqueness(self, df, table_name, key_columns):
        """Checks for uniqueness of primary key columns."""
        print(f"Running Uniqueness checks on {table_name}...")
        
        # Ensure key_columns is a list
        if isinstance(key_columns, str):
            key_columns = [key_columns]
 
        # Verify columns exist
        missing_cols = [c for c in key_columns if c not in df.columns]
        if missing_cols:
             self.log_result(table_name, f"Uniqueness Check - {key_columns}", "ERROR", f"Columns not found: {missing_cols}")
             return
 
        window_spec = df.groupBy(key_columns).count()
        duplicate_count = window_spec.filter(col("count") > 1).count()
        
        if duplicate_count > 0:
            self.log_result(table_name, f"Uniqueness Check - {key_columns}", "FAIL", f"Found {duplicate_count} duplicate keys", duplicate_count)
        else:
            self.log_result(table_name, f"Uniqueness Check - {key_columns}", "PASS", "Unique keys valid")
 
    def check_date_logic(self, df, table_name):
        """Checks logical consistency of dates (e.g., Term Date > Hire Date)."""
        print(f"Running Date Logic checks on {table_name}...")
        
        cols_lower = {c.lower(): c for c in df.columns}
        term_col = cols_lower.get("termination_date") or cols_lower.get("term_date")
        hire_col = cols_lower.get("hire_date")
        
        if term_col and hire_col:
            invalid_dates = df.filter(
                col(term_col).isNotNull() & 
                (col(term_col) < col(hire_col))
            ).count()
            
            if invalid_dates > 0:
                self.log_result(table_name, "Date Logic - Term > Hire", "FAIL", f"Found {invalid_dates} records where Term Date < Hire Date", invalid_dates)
            else:
                self.log_result(table_name, "Date Logic - Term > Hire", "PASS", "Date logic valid")
 
    def check_email_format(self, df, table_name, email_column="Email_Work"):
        """Validates email format."""
        print(f"Running Email Format checks on {table_name}...")
        actual_col = None
        for c in df.columns:
            if c.lower() == email_column.lower():
                actual_col = c
                break
        
        if actual_col is None:
             self.log_result(table_name, f"Email Format - {email_column}", "ERROR", f"Column {email_column} not found")
             return
 
        email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        invalid_emails = df.filter(
            col(actual_col).isNotNull() & 
            (col(actual_col) != "") &
            ~col(actual_col).rlike(email_pattern)
        ).count()
 
        if invalid_emails > 0:
            self.log_result(table_name, f"Email Format - {actual_col}", "FAIL", f"Found {invalid_emails} invalid email formats", invalid_emails)
        else:
            self.log_result(table_name, f"Email Format - {actual_col}", "PASS", "Email formats valid")
 
    def check_fte_range(self, df, table_name, fte_column="FTE"):
        """Checks if FTE is within 0.0 to 1.0 (or 100)."""
        print(f"Running FTE Range checks on {table_name}...")
        actual_col = None
        for c in df.columns:
            if c.lower() == fte_column.lower():
                actual_col = c
                break
        
        if not actual_col:
            return
 
        invalid_fte = df.filter(
            col(actual_col).isNotNull() &
            ((col(actual_col).cast("double") < 0.0) | (col(actual_col).cast("double") > 1.0))
        ).count()
 
        if invalid_fte > 0:
             self.log_result(table_name, f"FTE Range - {actual_col}", "FAIL", f"Found {invalid_fte} FTE values outside 0-1", invalid_fte)
        else:
             self.log_result(table_name, f"FTE Range - {actual_col}", "PASS", "FTE values valid")

    def save_logs_to_table(self, lakehouse_name=None):
        """Saves validation results to Delta tables dynamically based on the schema of the validated table."""
        if not self.results:
            print("No results to save.")
            return
 
        schema = StructType([
            StructField("run_id", StringType(), True),
            StructField("pipeline_name", StringType(), True),
            StructField("test_timestamp", TimestampType(), True),
            StructField("table_name", StringType(), True),
            StructField("test_check", StringType(), True),
            StructField("record_count", IntegerType(), True),
            StructField("message", StringType(), True),
            StructField("status", StringType(), True)
        ])
        
        # Group results by target schema/table for logging
        # We assume the log table is {Schema}.AuditLogs
        logs_by_schema = {}
        
        prefix = f"{lakehouse_name}." if lakehouse_name else ""
 
        for res in self.results:
            full_table_name = res["table_name"]
            
            # Attempt to parse schema from table name
            # Format usually: [Lakehouse.]Schema.Table
            parts = full_table_name.split('.')
            target_schema = None
            
            # Heuristic to find schema
            if len(parts) >= 2:
                # The table name is the last part, the schema is the second to last
                target_schema = parts[-2]
            else:
                print(f"Warning: Could not extract schema from {full_table_name}. Skipping.")
                continue
            
            # Construct the log table path
            log_table = f"{prefix}{target_schema}.AuditLogs"
            
            if log_table not in logs_by_schema:
                logs_by_schema[log_table] = []
            
            logs_by_schema[log_table].append(res)
            
        # Save batches
        for log_table_name, records in logs_by_schema.items():
            print(f"Saving {len(records)} logs to {log_table_name}...")
            try:
                df_log = self.spark.createDataFrame(records, schema)
                df_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table_name)
                print(f"Successfully saved to {log_table_name}.")
            except Exception as e:
                print(f"Error saving to {log_table_name}: {str(e)}")

In [ ]:
try:
    spark
except NameError:
    spark = SparkSession.builder.appName("People Data Quality").enableHiveSupport().getOrCreate()

# ==============================================================================
# PARAMETER HANDLING
# ==============================================================================
try:
    p_schema = schema if 'schema' in locals() else "Worker_HR"
    p_table = table if 'table' in locals() else "Worker"
    p_columns = columns if 'columns' in locals() else "Colleague_ID"
    p_run_id = run_id if 'run_id' in locals() else spark.conf.get("spark.fabric.runId", "MANUAL_RUN_" + datetime.now().strftime("%Y%m%d%H%M%S"))
except NameError:
    p_schema = "Worker_HR"
    p_table = "Worker"
    p_columns = "Colleague_ID"
    p_run_id = "MANUAL_RUN_" + datetime.now().strftime("%Y%m%d%H%M%S")

# Fallback/Alternative for Databricks widgets
try:
    from pyspark.dbutils import DBUtils
    dbutils = DBUtils(spark)
    try:
        p_schema = dbutils.widgets.get("schema")
        p_table = dbutils.widgets.get("table")
        p_columns = dbutils.widgets.get("columns")
        p_run_id = dbutils.widgets.get("run_id")
    except:
        pass
except ImportError:
    pass

# ==============================================================================
# EXECUTION
# ==============================================================================
PIPELINE_NAME = "Dynamic_Data_Quality_Check"
validator = PeopleDataQualityValidator(spark, p_run_id, PIPELINE_NAME)

full_table_name = f"{p_schema}.{p_table}"
columns_list = [c.strip() for c in p_columns.split(',')]

print(f"\n--- Starting checks for {full_table_name} ---")
print(f"Run ID: {p_run_id}")
print(f"Columns: {columns_list}")

try:
    df = spark.table(full_table_name)
    
    # 1. Null Checks
    validator.check_nulls(df, full_table_name, columns_list)
    
    # 2. Uniqueness Checks
    validator.check_uniqueness(df, full_table_name, columns_list)

except Exception as e:
    validator.log_result(full_table_name, "Load/Process Table", "ERROR", str(e))
    print(f"Error: {e}")

# Save Logs (Dynamically to {Schema}.AuditLogs)
LAKEHOUSE_NAME = None 
validator.save_logs_to_table(lakehouse_name=LAKEHOUSE_NAME)

# Display Summary
print("\n" + "="*50)
print("DATA QUALITY TEST SUMMARY")
print("="*50)

if validator.results:
    results_df = spark.createDataFrame(validator.results)
    results_df.show(truncate=False)
    
    failed_checks = [r for r in validator.results if r['status'] in ['FAIL', 'ERROR']]
    if failed_checks:
        print(f"\nWARNING: {len(failed_checks)} data quality checks failed.")
    else:
        print("\nSUCCESS: All data quality checks passed.")
else:
    print("No checks run or no results.")